# Online & Offline Feature Stores with Feast + MLflow (Iris Dataset)

This notebook builds a complete, small-scale feature store pipeline using **Feast** for feature management and **MLflow** for experiment tracking and model versioning, on the classic **Iris** dataset.

The two tools solve different problems and are commonly used together in production:

- **Feast** guarantees the *same feature definition* is used for training (offline) and serving (online) — this is what prevents training-serving skew.
- **MLflow** tracks *which model, trained on which parameters, achieved which metrics* — this is what makes a model's lineage reproducible and auditable.

## Plan

1. Load the Iris dataset and shape it into a feature table with entity IDs and timestamps
2. Define a Feast feature repository: an `Entity`, a `FeatureView`, offline (Parquet) and online (SQLite) stores
3. Register the definitions with Feast
4. Fetch **historical** features (offline store) and train a classifier, tracked with **MLflow**
5. **Materialize** features into the online store
6. Fetch **online** features to simulate a live prediction request
7. Load the tracked model back from MLflow and serve a prediction using the online features
8. Confirm the prediction matches what we'd expect from the offline-trained model — the whole point of keeping both stores in sync

## Step 0: Setup

- **Offline store**: a Parquet file (stand-in for a data warehouse)
- **Online store**: SQLite (stand-in for Redis/DynamoDB in production)
- **MLflow tracking store**: a local SQLite database (`mlflow.db`), since recent MLflow versions require a database backend rather than the older folder-based store

In [9]:
# Install dependencies (only needs to run once per environment)
%pip install feast mlflow scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(0)

REPO_PATH = "feature_repo"
os.makedirs(f"{REPO_PATH}/data", exist_ok=True)

print("Feature repo folder ready at:", os.path.abspath(REPO_PATH))

Feature repo folder ready at: C:\Shridhar\Study\BITS\Week 9\Demo\feature_repo


## Step 1: Load Iris and Shape It Into a Feature Table

Feast needs every row to have:
- An **entity id** — something to key features by (we use the row index as `iris_sample_id`)
- A **timestamp** — when this feature value became true (Iris has no natural timestamp, so we synthesize one, spacing samples 10 minutes apart as if they were collected over time)

We keep `target` / `species` alongside the features in the same Parquet file for convenience, but only the four measurement columns will be registered as *features* — the label is not something you'd normally serve from a feature store.

In [3]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
})

df["iris_sample_id"] = df.index
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

# Synthetic timestamps: pretend samples were collected 10 minutes apart
start = datetime.now() - timedelta(days=30)
df["event_timestamp"] = [start + timedelta(minutes=10 * i) for i in range(len(df))]
df["created"] = df["event_timestamp"]

print(df.shape)
df.head()

(150, 9)


,sepal_length,sepal_width,petal_length,petal_width,target,iris_sample_id,species,event_timestamp,created
0,5.1,3.5,1.4,0.2,0,0,setosa,2026-06-15 20:51:48.701815,2026-06-15 20:51:48.701815
1,4.9,3.0,1.4,0.2,0,1,setosa,2026-06-15 21:01:48.701815,2026-06-15 21:01:48.701815
2,4.7,3.2,1.3,0.2,0,2,setosa,2026-06-15 21:11:48.701815,2026-06-15 21:11:48.701815
3,4.6,3.1,1.5,0.2,0,3,setosa,2026-06-15 21:21:48.701815,2026-06-15 21:21:48.701815
4,5.0,3.6,1.4,0.2,0,4,setosa,2026-06-15 21:31:48.701815,2026-06-15 21:31:48.701815


In [4]:
# Save as Parquet -- this file plays the role of our offline store's data source
df.to_parquet(f"{REPO_PATH}/data/iris_features.parquet", index=False)
print("Saved offline source parquet file.")

Saved offline source parquet file.


## Step 2: Define the Feature Repository

Same pattern as any Feast project:

- An **Entity** (`iris_sample_id`) — what the features describe
- A **FileSource** — where the raw feature data lives
- A **FeatureView** — the four measurement features, tied to the entity and source, with a TTL

In [6]:
from datetime import timedelta
from feast import FeatureStore, Entity, FeatureView, Field, FileSource
from feast.types import Float32

iris_sample = Entity(name="iris_sample_id", description="Row identifier for an iris flower sample")

iris_source = FileSource(
    path=os.path.abspath(f"{REPO_PATH}/data/iris_features.parquet"),
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

iris_features_view = FeatureView(
    name="iris_features",
    entities=[iris_sample],
    ttl=timedelta(days=365),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
    ],
    source=iris_source,
    online=True,
)

print("Entity and feature view defined.")

Entity and feature view defined.


C:\Users\galan\AppData\Local\Temp\ipykernel_2644\537832930.py:5: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_sample_id'.
  iris_sample = Entity(name="iris_sample_id", description="Row identifier for an iris flower sample")


In [7]:
# Write feature_store.yaml -- tells Feast where the registry and online store live
yaml_content = f"""project: iris_feature_store
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
"""

with open(f"{REPO_PATH}/feature_store.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

project: iris_feature_store
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3



In [8]:
# Register (apply) our definitions -- equivalent to `feast apply` from the CLI
store = FeatureStore(repo_path=REPO_PATH)
store.apply([iris_sample, iris_features_view])

print("Registered feature views:")
for fv in store.list_feature_views():
    print(" -", fv.name, "->", [f.name for f in fv.features])

Registered feature views:
 - iris_features -> ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


## Step 3: Fetch Historical Features for Training

Point-in-time correct retrieval from the offline store. We pass an entity dataframe containing `iris_sample_id`, `event_timestamp`, and the label columns (`target`, `species`), and Feast joins in the feature values as of each timestamp.

In [10]:
entity_df = df[["iris_sample_id", "event_timestamp", "target", "species"]]

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

print("Training dataframe shape:", training_df.shape)
training_df.head()

Training dataframe shape: (150, 8)


,iris_sample_id,event_timestamp,target,species,sepal_length,sepal_width,petal_length,petal_width
0,0,2026-06-15 20:51:48.701815+00:00,0,setosa,5.1,3.5,1.4,0.2
1,1,2026-06-15 21:01:48.701815+00:00,0,setosa,4.9,3.0,1.4,0.2
2,2,2026-06-15 21:11:48.701815+00:00,0,setosa,4.7,3.2,1.3,0.2
3,3,2026-06-15 21:21:48.701815+00:00,0,setosa,4.6,3.1,1.5,0.2
4,4,2026-06-15 21:31:48.701815+00:00,0,setosa,5.0,3.6,1.4,0.2


## Step 4: Train a Model, Tracked with MLflow

We train a `RandomForestClassifier` on the offline features and log the run — parameters, metrics, and the model artifact itself — to MLflow. This is what makes the model's provenance reproducible: anyone can look up run `18318c04...` (or whatever ID this run gets) and see exactly what data shape, parameters, and code produced it.

In [14]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = training_df[feature_cols]
y = training_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MLflow requires a database-backed tracking store in recent versions;
# a local SQLite file works well for a self-contained notebook.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("iris_feast_demo")

with mlflow.start_run(run_name="rf_iris_feast") as run:
    n_estimators = 100
    max_depth = 4

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("feature_source", "feast:iris_features")

    model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    mlflow.log_metric("test_accuracy", acc)

    mlflow.sklearn.log_model(model, name="model")
    run_id = run.info.run_id

print("MLflow run ID:", run_id)
print("Test accuracy:", acc)

C:\Shridhar\Study\BITS\Week 9\Demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/15 20:52:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/15 20:52:06 INFO mlflow.store.db.utils: Updating database tables
2026/07/15 20:52:09 INFO mlflow.tracking.fluent: Experiment with name 'iris_feast_demo' does not exist. Creating a new experiment.


MLflow run ID: ed903ec1a1cb4c4b94b7a34660010c20
Test accuracy: 0.9666666666666667


## Step 5: Materialize Features into the Online Store

Push the latest feature values from the offline source into the online SQLite store, exactly as a scheduled materialization job would in production (against Redis/DynamoDB).

In [16]:
store.materialize_incremental(end_date=datetime.now())
print("Materialization complete.")

Materializing 1 feature views to 2026-07-15 20:52:40+00:00 into the sqlite online store.

iris_features from 2025-07-15 15:22:40+00:00 to 2026-07-15 20:52:40+00:00:
Materialization complete.


## Step 6: Fetch Online Features (Simulate a Live Request)

Pick a handful of samples and fetch their features from the online store — exactly what a real-time inference API would do given an `iris_sample_id`.

In [18]:
sample_ids = df["iris_sample_id"].sample(5, random_state=1).tolist()

online_features = store.get_online_features(
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
    entity_rows=[{"iris_sample_id": sid} for sid in sample_ids],
).to_df()

online_features

,iris_sample_id,petal_length,sepal_length,sepal_width,petal_width
0,14,1.2,5.8,4.0,0.2
1,98,3.0,5.1,2.5,1.1
2,75,4.4,6.6,3.0,1.4
3,16,1.3,5.4,3.9,0.4
4,131,6.4,7.9,3.8,2.0


## Step 7: Load the MLflow Model and Serve a Prediction

This is the full loop: a model trained on **offline** features, tracked in **MLflow**, now making predictions on **online** features fetched at serving time.

In [20]:
loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

X_serve = online_features[feature_cols]
predicted_class = loaded_model.predict(X_serve)

species_lookup = dict(enumerate(iris.target_names))
result = online_features[["iris_sample_id"]].copy()
result["predicted_species"] = [species_lookup[c] for c in predicted_class]

# Bring in the true species for comparison
true_species = df[df["iris_sample_id"].isin(sample_ids)][["iris_sample_id", "species"]]
result = result.merge(true_species, on="iris_sample_id")
result["correct"] = result["predicted_species"] == result["species"]

result

,iris_sample_id,predicted_species,species,correct
0,14,setosa,setosa,True
1,98,versicolor,versicolor,True
2,75,versicolor,versicolor,True
3,16,setosa,setosa,True
4,131,virginica,virginica,True


If `correct` is `True` across the board (or close to it — this is a held-out-style spot check, not the formal test set), the pipeline is working end-to-end: the model trained on offline Feast features generalizes correctly when fed the *same* features retrieved through the online path.

## Wrap-Up

In this notebook we:

- Shaped the Iris dataset into a Feast-compatible feature table with entity IDs and timestamps
- Defined a single feature view (`iris_features`) as the shared source of truth for four measurement features
- Retrieved point-in-time correct historical features and trained a `RandomForestClassifier`, with the full run tracked in MLflow (parameters, metrics, model artifact)
- Materialized features into an online store and fetched them again as a live service would
- Loaded the tracked model back from MLflow and served predictions on those online features

**Feast and MLflow are complementary, not overlapping**: Feast owns feature consistency and the offline/online split; MLflow owns model versioning, experiment comparison, and reproducibility. A production system typically uses both — this notebook is that pattern at the smallest scale that still exercises every part of it.